In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error

torch.manual_seed(42)
np.random.seed(42)

In [3]:
breastCancer = load_breast_cancer()
X_cls, y_cls = bc.data, bc.target

In [4]:
cal = fetch_california_housing()
X_reg, y_reg = cal.data, cal.target

In [5]:
def prepare_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test

In [6]:
Xc_train, Xc_test, yc_train, yc_test = prepare_data(X_cls, y_cls)
Xr_train, Xr_test, yr_train, yr_test = prepare_data(X_reg, y_reg)

In [7]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, depth, hidden_dim=64):
        super().__init__()
        layers = []

        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.ReLU())

        for _ in range(depth - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())

        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [8]:
def train_model(model, optimizer, criterion, X_train, y_train, epochs=100):
    losses = []

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()

        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return losses

In [9]:
def kfold_classification(X, y, depth, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    accs, losses = [], []

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        X_tr = torch.tensor(X_tr, dtype=torch.float32)
        y_tr = torch.tensor(y_tr, dtype=torch.long)
        X_val = torch.tensor(X_val, dtype=torch.float32)
        y_val = torch.tensor(y_val, dtype=torch.long)

        model = MLP(X.shape[1], 2, depth)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        train_model(model, optimizer, criterion, X_tr, y_tr)

        model.eval()
        with torch.no_grad():
            preds = torch.argmax(model(X_val), dim=1)
            accs.append(accuracy_score(y_val, preds))

    return np.mean(accs)

In [10]:
def kfold_regression(X, y, depth, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    mses = []

    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        X_tr = torch.tensor(X_tr, dtype=torch.float32)
        y_tr = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)
        X_val = torch.tensor(X_val, dtype=torch.float32)
        y_val = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

        model = MLP(X.shape[1], 1, depth)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()

        train_model(model, optimizer, criterion, X_tr, y_tr)

        model.eval()
        with torch.no_grad():
            preds = model(X_val)
            mses.append(mean_squared_error(y_val, preds))

    return np.mean(mses)

In [11]:
results_cls = []
results_reg = []

for depth in range(1, 5):
    acc = kfold_classification(Xc_train, yc_train, depth)
    mse = kfold_regression(Xr_train, yr_train, depth)

    results_cls.append({"Depth": depth, "CV_Accuracy": acc})
    results_reg.append({"Depth": depth, "CV_MSE": mse})

In [12]:
def test_classification(X_train, y_train, X_test, y_test, depth):
    model = MLP(X_train.shape[1], 2, depth)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    train_model(
        model, optimizer, criterion,
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )

    with torch.no_grad():
        preds = torch.argmax(
            model(torch.tensor(X_test, dtype=torch.float32)), dim=1
        )
    return accuracy_score(y_test, preds)

In [13]:
def test_regression(X_train, y_train, X_test, y_test, depth):
    model = MLP(X_train.shape[1], 1, depth)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    train_model(
        model, optimizer, criterion,
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    )

    with torch.no_grad():
        preds = model(torch.tensor(X_test, dtype=torch.float32))
    return mean_squared_error(y_test, preds)

In [14]:
for r in results_cls:
    r["Test_Accuracy"] = test_classification(
        Xc_train, yc_train, Xc_test, yc_test, r["Depth"]
    )

for r in results_reg:
    r["Test_MSE"] = test_regression(
        Xr_train, yr_train, Xr_test, yr_test, r["Depth"]
    )

df_cls = pd.DataFrame(results_cls)
df_reg = pd.DataFrame(results_reg)

df_cls, df_reg

(   Depth  CV_Accuracy  Test_Accuracy
 0      1     0.978824       0.979021
 1      2     0.971792       0.979021
 2      3     0.983529       0.979021
 3      4     0.978824       0.979021,
    Depth    CV_MSE  Test_MSE
 0      1  1.191511  0.991198
 1      2  0.674554  0.626134
 2      3  0.556010  0.524993
 3      4  0.505425  0.488660)